# Preconditions
`./setup_auth.ipynb` and `./setup_catalog_policies.ipynb`will run

In [73]:
%run ./setup_catalog_policies.ipynb

/Users/apabook/Desktop/ds-protocol/static/tutorial/venv/bin/python
{
    "participant_id": "did:jwk:provider",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2026-03-24T08:23:27.578761",
    "last_interaction": "2026-03-24T08:23:27.578823",
    "is_me": true
}
{
    "participant_id": "did:jwk:consumer",
    "participant_slug": "Myself",
    "participant_type": "Agent",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2026-03-24T08:23:27.551672",
    "last_interaction": "2026-03-24T08:23:27.551707",
    "is_me": true
}
Provider DID: did:jwk:provider

Provider token: token

Consumer DID: did:jwk:consumer

Consumer token: token
{
    "dctConformsTo": null,
    "dctCreator": null,
    "dctIdentifier": "urn:catalog:0689bcec-514f-48ca-b168-14ae10a84364",
    "dctIssued": "2026-03-24T08:23:27.656914Z",
    "dctModified": null,
    "dctTitle": null,
    "dspaceMainCatalo

# Contract Negotiation

## Initialization of negotiation request (Consumer -> Provider)

In [74]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request-init"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "18"
                }]
            }
        ]
    }
}




try:
    response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
    response_as_json = response.json()
    cn_consumer_id = response_as_json["response"]["consumerPid"]
    cn_provider_id = response_as_json["response"]["providerPid"]
    print(json.dumps(response_as_json, indent=2))
except Exception as e:
    print("Error in response, {}.".format(e))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "offer": {
      "@id": "urn:odrl-policy:afa59920-633a-475f-8932-5a000128a012",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "18",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:afdca292-6350-48b9-8108-0b6afbc1c53b"
    }
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:db2c

## Provider creates initial offer (Provider -> Consumer)

In [75]:
# Provider creates initial offer (Provider -> Consumer)
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use", 
                "constraint": [{
                    "leftOperand": "count", 
                    "operator": "gt", 
                    "rightOperand": "21"
                }]
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:afa59920-633a-475f-8932-5a000128a012",
      "permission": [
        {
          "action": "use",
          "constraint": [
            {
              "rightOperand": "21",
              "leftOperand": "count",
              "operator": "gt"
            }
          ]
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:afdca292-6350-48b9-8108-0b6afbc1c53b"
    },
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:78321a74-5008-459d-9d

## Consumer sends negotiation request based on offer (Consumer -> Provider)

In [76]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-request"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:afa59920-633a-475f-8932-5a000128a012",
      "permission": [
        {
          "action": "use"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:afdca292-6350-48b9-8108-0b6afbc1c53b"
    },
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "REQUESTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:db2c3406-cac5-409b-af80-db6c60bc4eb3",
    "state": "REQUESTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http

## Provider updates/confirms the offer (Provider -> Consumer)

In [77]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-offer"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "offer": {
      "@id": "urn:odrl-policy:afa59920-633a-475f-8932-5a000128a012",
      "permission": [
        {
          "action": "supermegause"
        }
      ],
      "@type": "Offer",
      "target": "urn:dataset:afdca292-6350-48b9-8108-0b6afbc1c53b"
    },
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "OFFERED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:78321a74-5008-459d-9dd8-0fe369727bf6",
    "state": "OFFERED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": 

## Consumer accepts the offer (Consumer -> Provider)

In [78]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-acceptance"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "ACCEPTED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:db2c3406-cac5-409b-af80-db6c60bc4eb3",
    "state": "ACCEPTED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T08:25:02.038960Z",
    "updatedAt": "2026-03-24T08:25:03.845979Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:9

## Provider creates the Agreement (Provider -> Consumer)

In [79]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-agreement"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "AGREED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:78321a74-5008-459d-9dd8-0fe369727bf6",
    "state": "AGREED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T08:25:01.948464Z",
    "updatedAt": "2026-03-24T08:25:04.184215Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:9e726

## Consumer verifies the agreement (Consumer -> Provider)

In [80]:
url = data_space_consumer + "/dsp/current/negotiations/rpc/setup-verification"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "VERIFIED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:db2c3406-cac5-409b-af80-db6c60bc4eb3",
    "state": "VERIFIED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T08:25:02.038960Z",
    "updatedAt": "2026-03-24T08:25:05.204666Z",
    "identifiers": {
      "providerPid": "urn:provider-pid:9

## Provider finalizes the negotiation (Provider -> Consumer)

In [81]:
url = data_space_provider + "/dsp/current/negotiations/rpc/setup-finalization"
payload = {
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
try:
    agreement = response_as_json
    agreement_id = response_as_json["negotiationAgentModel"]["agreement"]["id"]
    print(json.dumps(response_as_json, indent=2))
except KeyError:
    print("Error in response, check the payload and the endpoint.")
    print(json.dumps(response_as_json, indent=2))
    raise

{
  "request": {
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "consumerPid": "urn:consumer-pid:5d432400-b824-45ef-ad2b-48cfe031e59e",
    "providerPid": "urn:provider-pid:9e7268d7-cff5-42e3-a7c4-eaa6f2994300",
    "state": "FINALIZED"
  },
  "negotiationAgentModel": {
    "id": "urn:negotiation-process:78321a74-5008-459d-9dd8-0fe369727bf6",
    "state": "FINALIZED",
    "stateAttribute": null,
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "createdAt": "2026-03-24T08:25:01.948464Z",
    "updatedAt": "2026-03-24T08:25:06.004415Z",
    "identifiers": {
      "consumerPid": "urn:consumer-pid

## Final agreement

In [82]:
print("Final agreement: \n{}\n".format(json.dumps(agreement["negotiationAgentModel"]["agreement"], indent=2)))
print("Final agreement id: \n{}\n".format(agreement_id))

Final agreement: 
{
  "id": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
  "negotiationAgentProcessId": "urn:negotiation-process:78321a74-5008-459d-9dd8-0fe369727bf6",
  "negotiationAgentMessageId": "urn:negotiation-message:c82b48a8-205a-4606-b732-04ef5047accc",
  "consumerParticipantId": "did:jwk:consumer",
  "providerParticipantId": "did:jwk:provider",
  "agreementContent": {
    "@id": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
    "@type": "Agreement",
    "assignee": "did:jwk:consumer",
    "assigner": "did:jwk:provider",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "target": "urn:dataset:afdca292-6350-48b9-8108-0b6afbc1c53b",
    "timestamp": "1774340704"
  },
  "target": "urn:dataset:afdca292-6350-48b9-8108-0b6afbc1c53b",
  "state": "ACTIVE",
  "createdAt": "2026-03-24T08:25:04.189705Z",
  "updatedAt": "2026-03-24T08:25:06.012647Z"
}

Final agreement id: 
urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024



# Transfer Negotiation with Agents


## Transfer request setup (Consumer -> Provider)


In [93]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+asd",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["response"]["consumerPid"]
transfer_process_provider_pid = response_as_json["response"]["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "associatedAgentPeer": "did:jwk:provider",
    "agreementId": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
    "format": "http+asd",
    "dataAddress": null,
    "providerAddress": "http://127.0.0.1:1200/dsp/current",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "state": "REQUESTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:ac290d84-fb10-4970-8fa0-3e154ce98d8c",
    "state": "REQUESTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
    "c

## Start transfer (Provider -> Consumer)

In [94]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "state": "STARTED",
    "dataAddress": {
      "endpointType": "HttpProxy",
      "endpoint": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:94595146-940b-48c8-aa02-d5eb466c8118",
      "endpointProperties": null
    }
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f648058b-3328-4cc7-ba27-fd5e52290362",
    "state": "STARTED",
    "stateAttribute": "OnRequest",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "ag

## Suspend transfer (Consumer -> Provider)

In [95]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:ac290d84-fb10-4970-8fa0-3e154ce98d8c",
    "state": "SUSPENDED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
    "callbackAddress": "http://127.0.0.1:1200/dsp/cu

## Restart transfer (Consumer -> Provider)

In [96]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "state": "STARTED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:ac290d84-fb10-4970-8fa0-3e154ce98d8c",
    "state": "STARTED",
    "stateAttribute": "ByConsumer",
    "associatedAgentPeer": "did:jwk:provider",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
    "callbackAddress": "http://127.0.0.1:1200/dsp/current",
    "role": "Consumer",
    "properties": {},
    "errorDetails": null,
    "createdAt":

## Suspension by Provider (Provider -> Consumer)
(Note: Testing provider-initiated suspension)

In [97]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "state": "SUSPENDED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f648058b-3328-4cc7-ba27-fd5e52290362",
    "state": "SUSPENDED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
    "callbackAddress": "http://127.0.0.1:1100/dsp/cu

## Failure Test: Attempt start with invalid parameters

In [98]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735"
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "Petition Error\n"
    ]
  }
}


## Failure Test: Attempt duplicate or invalid suspension

In [99]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"]
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "code": "SUSPEND",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "error": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferError",
    "consumerPid": null,
    "providerPid": null,
    "code": "5000",
    "reason": [
      "Parse Error\n"
    ]
  }
}


## Finalize transfer (Provider -> Consumer)

In [100]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "request": {
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735"
  },
  "response": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "consumerPid": "urn:consumer-pid:f50dbacc-5d6c-4f39-b93e-d169a6d65fd4",
    "providerPid": "urn:provider-pid:4d42ec38-d017-4656-98aa-d1262ac8c735",
    "state": "COMPLETED"
  },
  "transferAgentModel": {
    "id": "urn:transfer-process:f648058b-3328-4cc7-ba27-fd5e52290362",
    "state": "COMPLETED",
    "stateAttribute": "ByProvider",
    "associatedAgentPeer": "did:jwk:consumer",
    "protocol": "DSP",
    "connectorInstanceId": "http+asd",
    "transferDirection": "Pull",
    "agreementId": "urn:agreement:fabd32f9-e746-4ccd-b717-abc6bbcd9024",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "role": "Provider",
    "properties": {},
    "errorDetails": null,
    "created